# Future MRT Pipeline

Attaches coordinates to planned MRT stations and builds a town-level developments lookup for the frontend.

**Step 1** — Match future MRT stations against the URA Master Plan 2025 GeoJSON to extract polygon centroids (lat/lon).  
**Step 2** — Combine with future transport hubs and group by town into `outputs/town_developments.json`.

**Inputs:**
- `data/future_mrt_stations.csv`
- `data/future_transport_hubs.csv`
- `data/MasterPlan2025RailStationLayer.geojson`

**Outputs:**
- `data/future_mrt_stations_with_coords.csv`
- `outputs/town_developments.json`
- `outputs/future_mrt_stations_for_frontend.csv`
- `outputs/future_transport_hubs_for_frontend.csv`

In [4]:
import json
import pandas as pd
from pathlib import Path
from shapely.geometry import shape

DATA_DIR = Path("../data")
OUTPUTS_DIR = Path("../outputs")

## Step 1 — Attach coordinates to future MRT stations

In [5]:
stations_df = pd.read_csv(DATA_DIR / "future_mrt_stations.csv")
print(f"Loaded {len(stations_df)} future MRT stations")

with open(DATA_DIR / "MasterPlan2025RailStationLayer.geojson") as f:
    geojson = json.load(f)
print(f"Loaded GeoJSON with {len(geojson['features'])} features")

# Build lookup: NAME -> first matching feature (some stations may have multiple polygons)
geojson_lookup = {}
for feature in geojson["features"]:
    name = feature["properties"]["NAME"].strip()
    if name not in geojson_lookup:
        geojson_lookup[name] = feature

print(f"Unique station names in GeoJSON: {len(geojson_lookup)}")

Loaded 54 future MRT stations
Loaded GeoJSON with 272 features
Unique station names in GeoJSON: 235


In [6]:
def get_centroid(feature):
    """Return (lat, lon) centroid of a GeoJSON feature polygon.
    GeoJSON coordinates are stored as [lon, lat], so centroid.x = lon, centroid.y = lat.
    """
    geom = shape(feature["geometry"])
    c = geom.centroid
    return c.y, c.x  # lat, lon


lats, lons = [], []
unmatched = []

for _, row in stations_df.iterrows():
    name_upper = str(row["station_name"]).strip().upper()

    feature = geojson_lookup.get(name_upper)
    if feature is None:
        feature = geojson_lookup.get(name_upper + " INTERCHANGE")

    if feature is None:
        print(f"WARNING: No GeoJSON match for '{row['station_name']}'")
        unmatched.append(row["station_name"])
        lats.append(None)
        lons.append(None)
    else:
        lat, lon = get_centroid(feature)
        lats.append(lat)
        lons.append(lon)

stations_df["lat"] = lats
stations_df["lon"] = lons

matched = stations_df["lat"].notna().sum()
print(f"\nMatched {matched}/{len(stations_df)} stations")
if unmatched:
    print(f"Unmatched ({len(unmatched)}): {unmatched}")


Matched 43/54 stations
Unmatched (11): ['Woodlands North (RTS)', 'Tengah Central', 'Nanyang', 'Taman Jurong', 'NTU', 'Boon Lay South', 'Gek Poh South', 'Bukit Batok South', 'Punggol North', 'Rivervale', 'Sungei Kadut']


In [7]:
out_csv = DATA_DIR / "future_mrt_stations_with_coords.csv"
stations_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
stations_df.head()

Saved: ..\data\future_mrt_stations_with_coords.csv


,station_name,line,line_code,town,expected_year,status,notes,lat,lon
0,Keppel,Circle Line,CCL,BUKIT MERAH,2026.0,Under construction,CCL Stage 6 - closes the Circle Line loop,1.270172,103.830296
1,Cantonment,Circle Line,CCL,BUKIT MERAH,2026.0,Under construction,CCL Stage 6,1.273082,103.836740
2,Prince Edward Road,Circle Line,CCL,BUKIT MERAH,2026.0,Under construction,CCL Stage 6,1.273241,103.847266
3,Bedok South,Thomson-East Coast Line,TEL,BEDOK,2026.0,Under construction,TEL Stage 5,1.316646,103.949230
4,Sungei Bedok,Thomson-East Coast Line,TEL,BEDOK,2026.0,Under construction,TEL Stage 5 - interchange with DTL,1.320396,103.957208


## Step 2 — Build town developments lookup

In [8]:
stations_coords = pd.read_csv(DATA_DIR / "future_mrt_stations_with_coords.csv")
hubs_df = pd.read_csv(DATA_DIR / "future_transport_hubs.csv")

print(f"Stations: {len(stations_coords)} rows across {stations_coords['town'].nunique()} towns")
print(f"Transport hubs: {len(hubs_df)} rows across {hubs_df['town'].nunique()} towns")

Stations: 54 rows across 20 towns
Transport hubs: 13 rows across 11 towns


In [9]:
def clean_val(val):
    """Convert NaN/NA to None; convert whole-number floats (e.g. 2027.0) to int."""
    if pd.isna(val):
        return None
    if isinstance(val, float) and val.is_integer():
        return int(val)
    return val


town_developments = {}

# Add MRT station entries
for town, group in stations_coords.groupby("town"):
    town_developments.setdefault(town, {"mrt_stations": [], "transport_hubs": []})
    for _, row in group.iterrows():
        town_developments[town]["mrt_stations"].append({
            "station_name": row["station_name"],
            "line": row["line"],
            "line_code": row["line_code"],
            "expected_year": clean_val(row["expected_year"]),
            "status": row["status"],
            "lat": clean_val(row["lat"]),
            "lon": clean_val(row["lon"]),
            "notes": clean_val(row["notes"]),
        })

# Add transport hub entries
for town, group in hubs_df.groupby("town"):
    town_developments.setdefault(town, {"mrt_stations": [], "transport_hubs": []})
    for _, row in group.iterrows():
        town_developments[town]["transport_hubs"].append({
            "hub_name": row["hub_name"],
            "hub_type": row["hub_type"],
            "expected_year": clean_val(row["expected_year"]),
            "status": row["status"],
            "notes": clean_val(row["notes"]),
        })

print(f"Total towns in output: {len(town_developments)}")
print(f"Towns with MRT data:  {sum(1 for v in town_developments.values() if v['mrt_stations'])}")
print(f"Towns with hub data:  {sum(1 for v in town_developments.values() if v['transport_hubs'])}")

Total towns in output: 22
Towns with MRT data:  20
Towns with hub data:  11


In [11]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Save JSON
out_json = OUTPUTS_DIR / "town_developments.json"
with open(out_json, "w") as f:
    json.dump(town_developments, f, indent=2)
print(f"Saved: {out_json}")

# Flatten to CSVs for easier frontend consumption
mrt_rows, hub_rows = [], []
for town, data in town_developments.items():
    for s in data["mrt_stations"]:
        mrt_rows.append({"town": town, **s})
    for h in data["transport_hubs"]:
        hub_rows.append({"town": town, **h})

mrt_csv_df = pd.DataFrame(mrt_rows, columns=["town", "station_name", "line", "line_code", "expected_year", "status", "lat", "lon", "notes"])
hubs_csv_df = pd.DataFrame(hub_rows, columns=["town", "hub_name", "hub_type", "expected_year", "status", "notes"])

mrt_csv_df.to_csv(OUTPUTS_DIR / "future_mrt_stations_for_frontend.csv", index=False)
hubs_csv_df.to_csv(OUTPUTS_DIR / "future_transport_hubs_for_frontend.csv", index=False)

print(f"Saved: future_mrt_stations_for_frontend.csv ({len(mrt_csv_df)} rows)")
print(f"Saved: future_transport_hubs_for_frontend.csv ({len(hubs_csv_df)} rows)")

Saved: ..\outputs\town_developments.json
Saved: future_mrt_stations_for_frontend.csv (54 rows)
Saved: future_transport_hubs_for_frontend.csv (13 rows)
